# 02 — Baseline results (TF-IDF + logistic regression)

CPU-only, so the complete transfer-gap picture exists before any GPU time is spent.

Run the sweep first:

```bash
python scripts/run_baseline_all.py
```

In [ ]:
import sys, pathlib, logging
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))
import pandas as pd, numpy as np
pd.set_option("display.width", 200)
from wcb.logging_utils import setup_logging
setup_logging(logging.WARNING)

In [ ]:
from wcb import analysis, config as C, results_io
runs = results_io.load_all("baseline")
print(f"{len(runs)} fold-evaluations   seeds={sorted(runs.seed.unique())}")
runs.groupby("condition")[["weighted_f1", "macro_f1"]].mean().round(4)

## The three conditions

`pooled` is the ceiling, `lobo` is the unseen-bank case, and `size_control` is trained on
a random subset the same size as a LOBO fold — it separates "less data" from "this
specific bank is missing".

In [ ]:
gaps = analysis.transfer_gap_table("baseline", n_boot=C.N_BOOTSTRAP)
summary = analysis.average_over_seeds(gaps)
summary[["pooled_weighted_f1", "size_control_weighted_f1", "lobo_weighted_f1",
         "gap_total", "gap_data", "gap_bank"]].round(4)

In [ ]:
print(f"mean transfer gap        {summary.gap_total.mean():+.4f}")
print(f"  from less data         {summary.gap_data.mean():+.4f}")
print(f"  from the bank missing  {summary.gap_bank.mean():+.4f}")
print(f"  share from bank identity: {summary.gap_bank.mean()/summary.gap_total.mean():.1%}")
print(f"positive gap for {(summary.gap_total>0).sum()}/{len(summary)} banks")

## Where does the drop land?

Per-class F1 by condition — the rare `irrelevant` class suffers most.

In [ ]:
(analysis.error_breakdown_by_class("baseline", C.PRIMARY_SEED)
   .groupby(["label", "condition"]).f1.mean().unstack()
   .loc[C.STANCE_LABELS].round(4))